# Lean-19d — Inférence bayésienne Beta-Binomial (C#)

**Navigation** : [Version Python](Lean-19d-Inference-Bayesian-Beta-Binomial-Python.ipynb) · [<< Lean-19 Hoeffding](Lean-19-Detection-MIMO-Index-Python.ipynb) · [Lean-19b Chernoff](Lean-19b-Borne-Chernoff-Python.ipynb) · [Lean-19c Bernstein](Lean-19c-Borne-Bernstein-CSharp.ipynb) · [Sommaire série](README.md)
## Contexte

Les carnets Lean-19 (Hoeffding), Lean-19b (Chernoff) et Lean-19c (Bernstein) établissent des **bornes fréquentistes** sur la probabilité de déviation entre la moyenne empirique `p_hat` et la vraie valeur `p` d'une loi de Bernoulli. Aucune information *a priori* sur `p` n'est utilisée.

L'inférence **bayésienne** procède autrement. On pose une loi *a priori* sur `p`, on observe les données, et on obtient une loi *a posteriori*. Pour la famille exponentielle Bernoulli, l'a priori conjugué est la **loi Beta** :

- **Priori** : `p ~ Beta(alpha, beta)`, paramètres `alpha > 0`, `beta > 0`.
- **Vraisemblance** : `X_1, ..., X_N ~ Bernoulli(p)` indépendantes, somme `S = sum(X_i)`.
- **Posteriori** : `p | data ~ Beta(alpha + S, beta + N - S)`.

Le posterior concentre toute l'information disponible sur `p` après observation. La **région de crédibilité** (HDI, *Highest Density Interval*) au niveau `credLevel` est l'intervalle de largeur minimale contenant `credLevel` de la masse a posteriori.

## Comparaison avec les bornes fréquentistes

Pour un BSC `Bernoulli(p)` avec `N` observations, la largeur du HDI bayésien dépend de l'a priori choisi. Deux choix classiques :

- **Beta(1, 1)** = loi uniforme sur `[0, 1]` = a priori non-informatif au sens où tout `p` est équiprobable.
- **Beta(0.5, 0.5)** = a priori de Jeffreys, piquant aux bords `p=0` et `p=1` (anti-pooling).

Asymptotiquement (`N -> oo`), le rapport `largeur HDI / largeur Hoeffding` **ne tend pas vers 1** : il converge vers une constante qui ne dépend que de `p` — environ `0.72` pour `p = 0.5` et `0.31` pour `p = 0.05`, soit un HDI respectivement ~30 % et ~3x plus serré. L'écart **ne se referme donc jamais** : Hoeffding est une borne *distribution-free*, tandis que le HDI hérite de la variance `p(1-p)` du posterior. C'est le choix de l'a priori qui déplace la largeur, sans changer cet ordre.

### Parité avec le carnet Python

Les bornes déterministes et la colonne des largeurs de Hoeffding sont identiques à celles du carnet Python à la précision affichée. Les statistiques qui dépendent des tirages sont recalculées avec `new Random(42)` ; elles peuvent différer de celles obtenues avec le générateur PCG64 de NumPy. Les références Python sont signalées comme citées. Pour chacun des deux a priori, une assertion vérifie sur chaque instance que le ratio HDI/Hoeffding est strictement inférieur à 1. La lecture des résultats cite les valeurs calculées en C#.

## Plan

1. E1 — `BetaPosteriorParams` : paramètres du posterior conjugué.
2. E2 — `BetaHdi` : intervalle de crédibilité par recherche sur la densité.
3. E3 — `BayesianVsFrequentistWidth` : compare largeur HDI vs Hoeffding.
4. Mesure sur 20 instances BSC (mêmes seed=42 que Lean-19/19b/19c pour continuité).

In [1]:
// Lean-19d — Setup
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

const int Seed = 42;
var rng = new Random(Seed); // même seed que Lean-19/19b/19c pour continuité

// La culture d'affichage ne dépend pas de la configuration du noyau.
void WriteInvariant(FormattableString message) =>
    Console.WriteLine(message.ToString(CultureInfo.InvariantCulture));

void Require(bool condition, FormattableString message)
{
    if (!condition)
        throw new InvalidOperationException(message.ToString(CultureInfo.InvariantCulture));
}

// Rapporte l'écart absolu et le verdict avant de faire échouer la cellule si nécessaire.
double CheckClose(string label, double actual, double expected, double tolerance)
{
    double error = Math.Abs(actual - expected);
    bool ok = double.IsFinite(error) && error <= tolerance;
    WriteInvariant($"{label} : calculé={actual:G}, référence Python citée={expected:G}, écart absolu={error:E3}, tolérance={tolerance:E3} — {(ok ? "CONFORME" : "ÉCART")}");
    Require(ok, $"{label} : écart {error:E3} supérieur à la tolérance {tolerance:E3} ou non fini.");
    return error;
}

## Exemple guidé 1 — Posterior conjugué Beta-Binomial

Pour un a priori `Beta(alpha, beta)` et `S` succès sur `N` essais bernoulli, le posterior est `Beta(alpha + S, beta + N - S)`.

Chaque succès ajoute 1 à `alpha` et chaque échec ajoute 1 à `beta` : le posterior reste une loi Beta, c'est la **conjugaison** de la famille Beta avec la vraisemblance de Bernoulli. Aucun calcul d'intégrale n'est nécessaire — la cellule suivante se contente d'additionner les paramètres, puis vérifie le résultat sur un petit exemple avec des appels à `CheckClose` avant de l'afficher.

In [2]:
// Exemple guidé 1 — Posterior conjugué Beta-Binomial
(double Alpha, double Beta) BetaPosteriorParams(
    double priorAlpha, double priorBeta, int successes, int failures)
{
    // Prior : p ~ Beta(priorAlpha, priorBeta).
    // Données : successes succès et failures échecs (successes + failures = N).
    // Posterior : p | data ~ Beta(priorAlpha + successes, priorBeta + failures).
    return (priorAlpha + successes, priorBeta + failures);
}

// Smoke test
// Additions exactes pour cet exemple ; seuil de comparaison repris de la source.
const double PosteriorTolerance = 1e-12;
var posterior = BetaPosteriorParams(1.0, 1.0, successes: 7, failures: 3);
// Références citées : sortie de la cellule 3 du carnet Python.
double posteriorAlphaError = CheckClose("Posterior α", posterior.Alpha, 8.0, PosteriorTolerance);
double posteriorBetaError = CheckClose("Posterior β", posterior.Beta, 4.0, PosteriorTolerance);
double posteriorMaxError = Math.Max(posteriorAlphaError, posteriorBetaError);
WriteInvariant($"Posterior: Beta({posterior.Alpha:F1}, {posterior.Beta:F1})");

Posterior α : calculé=8, référence Python citée=8, écart absolu=0.000E+000, tolérance=1.000E-012 — CONFORME


Posterior β : calculé=4, référence Python citée=4, écart absolu=0.000E+000, tolérance=1.000E-012 — CONFORME


Posterior: Beta(8.0, 4.0)


### Lecture du résultat : posterior Beta(8, 4)

La sortie affiche `Posterior: Beta(8.0, 4.0)` : avec l'a priori uniforme `Beta(1, 1)` et 7 succès pour 3 échecs (N = 10), les paramètres a posteriori valent 1 + 7 = 8 et 1 + 3 = 4. La conjugaison garde le posterior dans la famille Beta, et l'a priori n'apporte que 2 pseudo-observations contre 10 observations réelles : la moyenne a posteriori 8/12 ≈ 0,667 reste proche de la fréquence observée 7/10 = 0,7.

## Exemple guidé 2 — Intervalle de crédibilité HDI

Le HDI est l'intervalle `[a, b]` de largeur `b - a` minimale tel que `P(a <= p <= b | data) >= credLevel`. On le trouve par recherche numérique : pour un niveau `credLevel` et un posterior `Beta(alpha, beta)`, on balaie un seuil `t` et on prend l'intervalle des points où la densité excède `t`.

Contrairement à l'intervalle « equal-tailed » (quantiles 2,5 % et 97,5 %), le HDI ne suppose pas la symétrie : pour une densité unimodale comme celle d'une loi Beta, c'est le super-niveau {x : Pdf(x) ≥ t} de masse `credLevel`. La cellule suivante retrouve ce seuil `t` par dichotomie (40 itérations) sur une grille de 2001 points de [0, 1].

In [3]:
// Exemple guidé 2 — Intervalle de crédibilité HDI
class BetaDistribution
{
    private readonly double alpha;
    private readonly double beta;
    private readonly double logBeta;

    // Coefficients de Lanczos avec g = 7, pour calculer log Γ sans former Γ.
    private static readonly double[] Lanczos = {
        676.5203681218851, -1259.1392167224028, 771.3234287776531,
        -176.6150291621406, 12.507343278686905, -0.13857109526572012,
        9.984369578019572e-6, 1.5056327351493116e-7
    };

    public BetaDistribution(double alpha, double beta)
    {
        if (!double.IsFinite(alpha) || alpha <= 0.0)
            throw new ArgumentOutOfRangeException(nameof(alpha));
        if (!double.IsFinite(beta) || beta <= 0.0)
            throw new ArgumentOutOfRangeException(nameof(beta));
        this.alpha = alpha;
        this.beta = beta;
        logBeta = LogGamma(alpha) + LogGamma(beta) - LogGamma(alpha + beta);
    }

    public double Mean => alpha / (alpha + beta);

    private static double LogGamma(double z)
    {
        // Réflexion pour les formes inférieures à 0,5.
        if (z < 0.5)
            return Math.Log(Math.PI) - Math.Log(Math.Sin(Math.PI * z)) - LogGamma(1.0 - z);

        z -= 1.0;
        double sum = 0.99999999999980993;
        for (int i = 0; i < Lanczos.Length; i++)
            sum += Lanczos[i] / (z + i + 1.0);
        double t = z + 7.5;
        return 0.5 * Math.Log(2.0 * Math.PI) + (z + 0.5) * Math.Log(t) - t + Math.Log(sum);
    }

    public double Pdf(double x)
    {
        if (x < 0.0 || x > 1.0)
            return 0.0;
        // Limites aux bords : éviter notamment la forme 0 × log(0).
        if (x == 0.0)
            return alpha < 1.0 ? double.PositiveInfinity : alpha == 1.0 ? beta : 0.0;
        if (x == 1.0)
            return beta < 1.0 ? double.PositiveInfinity : beta == 1.0 ? alpha : 0.0;
        return Math.Exp((alpha - 1.0) * Math.Log(x) + (beta - 1.0) * Math.Log(1.0 - x) - logBeta);
    }

    public double Cdf(double x)
    {
        if (x <= 0.0) return 0.0;
        if (x >= 1.0) return 1.0;

        // Bêta incomplète régularisée : le préfacteur reste en log jusqu'à l'exponentiation.
        double front = Math.Exp(alpha * Math.Log(x) + beta * Math.Log(1.0 - x) - logBeta);
        // La symétrie permet d'évaluer la fraction continue du côté qui converge le mieux.
        if (x < (alpha + 1.0) / (alpha + beta + 2.0))
            return front * BetaFraction(alpha, beta, x) / alpha;
        return 1.0 - front * BetaFraction(beta, alpha, 1.0 - x) / beta;
    }

    private static double BetaFraction(double a, double b, double x)
    {
        // Méthode de Lentz modifiée : précision relative de 3e-14 sur le facteur multiplicatif.
        const double fractionTolerance = 3e-14;
        const double tiny = 1e-300; // Protège les inversions sans masquer la non-convergence.
        const int maxIterations = 512;
        static double Protect(double value) =>
            Math.Abs(value) < tiny ? Math.CopySign(tiny, value) : value;

        double c = 1.0;
        double d = 1.0 / Protect(1.0 - (a + b) * x / (a + 1.0));
        double fraction = d;
        for (int m = 1; m <= maxIterations; m++)
        {
            // Deux coefficients successifs de la fraction continue.
            double twiceM = 2.0 * m;
            double coefficient = m * (b - m) * x / ((a + twiceM - 1.0) * (a + twiceM));
            d = 1.0 / Protect(1.0 + coefficient * d);
            c = Protect(1.0 + coefficient / c);
            fraction *= d * c;

            coefficient = -(a + m) * (a + b + m) * x / ((a + twiceM) * (a + twiceM + 1.0));
            d = 1.0 / Protect(1.0 + coefficient * d);
            c = Protect(1.0 + coefficient / c);
            double factor = d * c;
            fraction *= factor;
            if (Math.Abs(factor - 1.0) <= fractionTolerance)
                return fraction;
        }
        throw new ArithmeticException("La fraction continue de la bêta incomplète n'a pas convergé.");
    }
}

(double Lower, double Upper) BetaHdi(
    double priorAlpha, double priorBeta, int successes, int failures, double credLevel = 0.95)
{
    var parameters = BetaPosteriorParams(priorAlpha, priorBeta, successes, failures);
    var rv = new BetaDistribution(parameters.Alpha, parameters.Beta);

    // Recherche d'un seuil t tel que l'intervalle des x : pdf(x) > t
    // a une masse exactement cred_level. Le HDI est l'intervalle de
    // niveau cred_level le plus étroit pour cette densité unimodale.
    // Échantillonnage de la densité pour trouver les bornes
    const int gridSize = 2001;
    const double gridStep = 1.0 / (gridSize - 1);
    var xs = new double[gridSize];
    var ys = new double[gridSize];
    for (int i = 0; i < gridSize; i++)
    {
        xs[i] = i * gridStep;
        ys[i] = rv.Pdf(xs[i]);
    }

    (double Lower, double Upper) IntervalAtThreshold(double threshold)
    {
        int first = -1;
        int last = -1;
        for (int i = 0; i < gridSize; i++)
        {
            if (ys[i] >= threshold)
            {
                if (first < 0) first = i;
                last = i;
            }
        }
        return first < 0 ? (0.0, 1.0) : (xs[first], xs[last]);
    }

    double MassOfInterval(double lower, double upper) => rv.Cdf(upper) - rv.Cdf(lower);

    // Dichotomie sur t
    double lo = 0.0;
    // « hi au-dessus de la densité max » : la formule initiale de la source n'est pas un majorant garanti.
    double hi = rv.Pdf(rv.Mean) * 1.5;
    for (int i = 0; i < 40; i++)
    {
        double mid = (lo + hi) / 2.0;
        var interval = IntervalAtThreshold(mid);
        double currentMass = MassOfInterval(interval.Lower, interval.Upper);
        if (currentMass > credLevel)
            lo = mid; // monter le seuil -> intervalle plus étroit
        else
            hi = mid; // baisser le seuil -> intervalle plus large
    }
    return IntervalAtThreshold(lo);
}

// Smoke test
// La sortie Python citée est arrondie à quatre décimales ; seuil absolu de parité fixé à 1e-4.
const double HdiParityTolerance = 1e-4;
const double CredLevel = 0.95;
const double SourceMassFloor = 0.94;
var hdi = BetaHdi(1.0, 1.0, successes: 7, failures: 3, credLevel: CredLevel);
var hdiPosterior = new BetaDistribution(posterior.Alpha, posterior.Beta);
double hdiWidth = hdi.Upper - hdi.Lower;
double hdiMass = hdiPosterior.Cdf(hdi.Upper) - hdiPosterior.Cdf(hdi.Lower);
WriteInvariant($"HDI {100 * CredLevel:F0}%: [{hdi.Lower:F4}, {hdi.Upper:F4}], largeur={hdiWidth:F4}, masse={hdiMass:F4}");
Require(hdiMass >= SourceMassFloor, $"Masse {hdiMass:F4} hors cible (seuil : {SourceMassFloor:F2}).");
// Références citées : sortie de la cellule 6 du carnet Python.
double hdiMaxError = new[] {
    CheckClose("HDI borne basse", hdi.Lower, 0.4115, HdiParityTolerance),
    CheckClose("HDI borne haute", hdi.Upper, 0.9065, HdiParityTolerance),
    CheckClose("HDI largeur", hdiWidth, 0.4950, HdiParityTolerance),
    CheckClose("HDI masse", hdiMass, 0.9502, HdiParityTolerance)
}.Max();

HDI 95%: [0.4115, 0.9065], largeur=0.4950, masse=0.9502


HDI borne basse : calculé=0.41150000000000003, référence Python citée=0.4115, écart absolu=5.551E-017, tolérance=1.000E-004 — CONFORME


HDI borne haute : calculé=0.9065, référence Python citée=0.9065, écart absolu=0.000E+000, tolérance=1.000E-004 — CONFORME


HDI largeur : calculé=0.49499999999999994, référence Python citée=0.495, écart absolu=5.551E-017, tolérance=1.000E-004 — CONFORME


HDI masse : calculé=0.9502260455429045, référence Python citée=0.9502, écart absolu=2.605E-005, tolérance=1.000E-004 — CONFORME


### Lecture du résultat : HDI 95 % = [0.4115, 0.9065]

La sortie affiche `HDI 95%: [0.4115, 0.9065], largeur=0.4950, masse=0.9502` : l'intervalle concentre 95,02 % de la masse a posteriori, un peu au-dessus du niveau 0,95 demandé — la dichotomie sur le seuil de densité vise une masse au moins égale au niveau requis, et l'appel à `Require` de la cellule vérifie `hdiMass >= 0.94`. À largeur 0,4950, c'est l'intervalle le plus étroit possible à ce niveau pour le posterior Beta(8, 4) : c'est la définition du HDI, pas un intervalle symétrique autour de la moyenne.

## Exemple guidé 3 — Comparaison HDI vs Hoeffding

Hoeffding donne un intervalle de **confiance** `p_hat ± eps` avec `eps = sqrt(ln(2/(1-c)) / (2N))` où `c` est le niveau de confiance (`c = 0.95` ici) — soit `ln(2/0.05) = ln(40) ≈ 3.689`, d'où `eps ≈ sqrt(3.689 / (2N))` ; en notant `alpha = 1 - c = 0.05` le niveau de risque, cela s'écrit aussi `eps = sqrt(ln(2/alpha) / (2N))`. Le HDI bayésien donne un intervalle de **crédibilité** calculé à partir de la loi a posteriori. Pour un BSC avec `S` succès sur `N`, on compare les deux largeurs.

Point clé : la largeur de Hoeffding `2·eps` ne dépend **que de N** (et du niveau de confiance), jamais de `p` — c'est le prix du « distribution-free ». Le HDI, lui, est calculé sur le posterior issu des données observées : sa largeur hérite de la variance `p(1-p)` estimée par le posterior. La cellule suivante tire une réalisation `S ~ Binomial(N, p_true)` avec le générateur `rng` (`new Random(42)`) et compare les deux largeurs sur cette instance.

In [4]:
// Illustration 3 — Comparaison HDI vs Hoeffding
// Largeur brute de l'intervalle, sans troncature aux bornes 0 et 1.
double HoeffdingWidth(int n, double confLevel = 0.95) =>
    2.0 * Math.Sqrt(Math.Log(2.0 / (1.0 - confLevel)) / (2.0 * n));

(int Successes, double HdiWidth, double Hoeffding, double Ratio) BayesianVsFrequentistWidth(
    int n, double pTrue, double priorAlpha, double priorBeta, double confLevel = 0.95)
{
    // Une binomiale est la somme de n essais de Bernoulli indépendants.
    int successes = 0;
    for (int i = 0; i < n; i++)
        if (rng.NextDouble() < pTrue) successes++;

    var interval = BetaHdi(priorAlpha, priorBeta, successes, n - successes, confLevel);
    double width = interval.Upper - interval.Lower;
    double bound = HoeffdingWidth(n, confLevel);
    return (successes, width, bound, width / bound);
}

// Budget déclaré avant les comparaisons : cinq écarts-types de la différence
// de deux réalisations indépendantes, plus le demi-pas d'arrondi de la référence.
// C'est une tolérance empirique, et non une garantie asymptotique de couverture.
const double McSigmaFactor = 5.0;
const int McReplicates = 384;
const double HalfUnitF4 = 0.00005;
const double HalfUnitF3 = 0.0005;

double SampleStdDev(IEnumerable<double> values)
{
    var data = values.ToArray();
    double mean = data.Average();
    return Math.Sqrt(data.Sum(x => (x - mean) * (x - mean)) / (data.Length - 1));
}

double McTolerance(IEnumerable<double> values, double rounding) =>
    McSigmaFactor * Math.Sqrt(2.0) * SampleStdDev(values) + rounding;

Func<Random, double> MakeRatioSampler(int n, double p, double priorA, double priorB, double level)
{
    // Calibration séparée : CDF binomiale par récurrence autour du mode,
    // avec un poids au mode fixé à 1 avant normalisation.
    int mode = (int)Math.Floor((n + 1.0) * p);
    var cdf = new double[n + 1];
    cdf[mode] = 1.0;
    for (int s = mode; s > 0; s--)
        cdf[s - 1] = cdf[s] * (s / (n - s + 1.0)) * ((1.0 - p) / p);
    for (int s = mode; s < n; s++)
        cdf[s + 1] = cdf[s] * ((n - s) / (s + 1.0)) * (p / (1.0 - p));
    for (int s = 1; s <= n; s++) cdf[s] += cdf[s - 1];
    double total = cdf[n];
    for (int s = 0; s <= n; s++) cdf[s] /= total;

    double bound = HoeffdingWidth(n, level);
    var ratiosByCount = new Dictionary<int, double>();
    return generator =>
    {
        // L'inversion de la CDF n'utilise pas le flux principal ; le cache
        // évite de recalculer le HDI lorsqu'un compte de succès se répète.
        double u = 1.0 - generator.NextDouble();
        int lo = 0;
        int hi = n;
        while (lo < hi)
        {
            int mid = lo + (hi - lo) / 2;
            if (cdf[mid] < u) lo = mid + 1;
            else hi = mid;
        }
        if (!ratiosByCount.TryGetValue(lo, out double ratio))
        {
            var interval = BetaHdi(priorA, priorB, lo, n - lo, level);
            ratio = (interval.Upper - interval.Lower) / bound;
            ratiosByCount.Add(lo, ratio);
        }
        return ratio;
    };
}

// Smoke test
const int SmokeN = 100;
const double SmokeP = 0.3;
var smokeCalibrationRng = new Random(Seed + 1);
var smokeSampler = MakeRatioSampler(SmokeN, SmokeP, 1.0, 1.0, CredLevel);
var smokeRatios = Enumerable.Range(0, McReplicates)
    .Select(_ => smokeSampler(smokeCalibrationRng)).ToArray();
double smokeBound = HoeffdingWidth(SmokeN, CredLevel);
double smokeWidthTolerance = McTolerance(smokeRatios.Select(x => x * smokeBound), HalfUnitF4);
double smokeRatioTolerance = McTolerance(smokeRatios, HalfUnitF3);

var smoke = BayesianVsFrequentistWidth(SmokeN, SmokeP, 1.0, 1.0, CredLevel);
WriteInvariant($"HDI={smoke.HdiWidth:F4}, Hoeffding={smoke.Hoeffding:F4}, ratio={smoke.Ratio:F3}");
Require(smoke.Successes >= 0 && smoke.Successes <= SmokeN, $"Nombre de succès hors domaine : {smoke.Successes}.");
Require(smoke.Ratio > 0.0 && smoke.Ratio < 1.0, $"Ratio HDI/Hoeffding hors de ]0, 1[ : {smoke.Ratio:F3}.");
// Références citées : sortie de la cellule 9 du carnet Python.
double smokeHoeffError = CheckClose("Exemple 3 — Hoeffding", smoke.Hoeffding, 0.2716, HalfUnitF4);
double smokeHdiError = CheckClose("Exemple 3 — largeur HDI", smoke.HdiWidth, 0.1815, smokeWidthTolerance);
double smokeRatioError = CheckClose("Exemple 3 — ratio", smoke.Ratio, 0.668, smokeRatioTolerance);
WriteInvariant($"Lecture C# : à N={SmokeN}, p={SmokeP:F1}, le HDI est {(1.0 - smoke.Ratio):P0} plus serré que l'intervalle de Hoeffding.");

HDI=0.1890, Hoeffding=0.2716, ratio=0.696


Exemple 3 — Hoeffding : calculé=0.27162030314812385, référence Python citée=0.2716, écart absolu=2.030E-005, tolérance=5.000E-005 — CONFORME


Exemple 3 — largeur HDI : calculé=0.189, référence Python citée=0.1815, écart absolu=7.500E-003, tolérance=5.482E-002 — CONFORME


Exemple 3 — ratio : calculé=0.6958242731101432, référence Python citée=0.668, écart absolu=2.782E-002, tolérance=2.021E-001 — CONFORME


Lecture C# : à N=100, p=0.3, le HDI est 30 % plus serré que l'intervalle de Hoeffding.


### Lecture du résultat : ratio HDI/Hoeffding

Pour N = 100, p_true = 0,3 et l'a priori `Beta(1, 1)`, la largeur Hoeffding 0,2716 = 2·sqrt(ln(40)/200) ne dépend que de N et du niveau de confiance, tandis que le HDI affiché est calculé sur le posterior issu du tirage `Binomial(100, 0.3)` effectué avec `rng` (`new Random(42)`). Le ratio affiché est inférieur à 1 : sur cette instance, l'intervalle de crédibilité est plus serré que l'intervalle de confiance de Hoeffding. La ligne « Lecture C# » chiffre cette réduction pour le tirage du port.

## Mesure — 20 instances BSC

Même protocole que Lean-19/19b/19c : 20 instances BSC avec seed=42, p variant sur `[0.05, 0.5]`, `N` variant sur `[100, 500, 1000, 5000]`. On compare la largeur HDI pour Beta(1,1) (uniforme) et Beta(0.5, 0.5) (Jeffreys) à la largeur Hoeffding.

Protocole : la cellule réinitialise `rng` avec `new Random(42)` au début de la mesure. Pour chaque couple (p, N) de la grille, elle tire deux nombres de succès successifs `S ~ Binomial(N, p)`, un pour chaque a priori, calcule les deux HDI et la largeur de Hoeffding, puis imprime le tableau `p, N, HDI_unif, Hoeff, R_unif, R_jeff` suivi des ratios médian et minimal agrégés sur les 20 instances.

In [5]:
// 20 instances BSC, mêmes seed/p/N que Lean-19/19b/19c
rng = new Random(Seed);
var ps = new[] { 0.05, 0.1, 0.2, 0.3, 0.5 };
var ns = new[] { 100, 500, 1000, 5000 };
var instances = (from p in ps from n in ns select (P: p, N: n)).ToArray(); // 20 instances
const int ExpectedInstances = 20;

// Références CITÉES : valeurs affichées par la cellule 12 du carnet Python.
// Elles ne fournissent ni comptes de succès ni données au calcul des HDI C#.
var cited = new (double P, int N, double HdiUnif, double Hoeff, double RUnif, double RJeff)[]
{
    (0.05,  100, 0.1005, 0.2716, 0.370, 0.309),
    (0.05,  500, 0.0420, 0.1215, 0.346, 0.325),
    (0.05, 1000, 0.0250, 0.0859, 0.291, 0.309),
    (0.05, 5000, 0.0120, 0.0384, 0.312, 0.325),
    (0.10,  100, 0.1125, 0.2716, 0.414, 0.387),
    (0.10,  500, 0.0510, 0.1215, 0.420, 0.457),
    (0.10, 1000, 0.0400, 0.0859, 0.466, 0.448),
    (0.10, 5000, 0.0175, 0.0384, 0.456, 0.430),
    (0.20,  100, 0.1630, 0.2716, 0.600, 0.641),
    (0.20,  500, 0.0710, 0.1215, 0.584, 0.597),
    (0.20, 1000, 0.0490, 0.0859, 0.570, 0.570),
    (0.20, 5000, 0.0225, 0.0384, 0.586, 0.586),
    (0.30,  100, 0.1830, 0.2716, 0.674, 0.676),
    (0.30,  500, 0.0815, 0.1215, 0.671, 0.650),
    (0.30, 1000, 0.0575, 0.0859, 0.669, 0.664),
    (0.30, 5000, 0.0255, 0.0384, 0.664, 0.677),
    (0.50,  100, 0.1925, 0.2716, 0.709, 0.712),
    (0.50,  500, 0.0875, 0.1215, 0.720, 0.720),
    (0.50, 1000, 0.0620, 0.0859, 0.722, 0.722),
    (0.50, 5000, 0.0280, 0.0384, 0.729, 0.729)
};
Require(instances.Length == ExpectedInstances && cited.Length == ExpectedInstances,
    $"Tailles incompatibles : grille={instances.Length}, référence={cited.Length}, attendu={ExpectedInstances}.");

double Median(IEnumerable<double> values)
{
    var ordered = values.OrderBy(x => x).ToArray();
    int mid = ordered.Length / 2;
    return ordered.Length % 2 == 0 ? (ordered[mid - 1] + ordered[mid]) / 2.0 : ordered[mid];
}

// La calibration répète la grille entière sur un flux distinct. Les médianes
// et les minima ont leurs propres dispersions, calculées sur ces grilles.
var gridCalibrationRng = new Random(Seed + 2);
var sampleUnif = instances.Select(i => MakeRatioSampler(i.N, i.P, 1.0, 1.0, CredLevel)).ToArray();
var sampleJeff = instances.Select(i => MakeRatioSampler(i.N, i.P, 0.5, 0.5, CredLevel)).ToArray();
var drawsUnif = instances.Select(_ => new double[McReplicates]).ToArray();
var drawsJeff = instances.Select(_ => new double[McReplicates]).ToArray();
var medianDrawsUnif = new double[McReplicates];
var medianDrawsJeff = new double[McReplicates];
var minDrawsUnif = new double[McReplicates];
var minDrawsJeff = new double[McReplicates];
var trialUnif = new double[ExpectedInstances];
var trialJeff = new double[ExpectedInstances];
for (int rep = 0; rep < McReplicates; rep++)
{
    for (int i = 0; i < instances.Length; i++)
    {
        drawsUnif[i][rep] = trialUnif[i] = sampleUnif[i](gridCalibrationRng);
        drawsJeff[i][rep] = trialJeff[i] = sampleJeff[i](gridCalibrationRng);
    }
    medianDrawsUnif[rep] = Median(trialUnif);
    medianDrawsJeff[rep] = Median(trialJeff);
    minDrawsUnif[rep] = trialUnif.Min();
    minDrawsJeff[rep] = trialJeff.Min();
}

var tolerances = instances.Select((instance, i) =>
{
    double bound = HoeffdingWidth(instance.N, CredLevel);
    return (
        Hdi: McTolerance(drawsUnif[i].Select(x => x * bound), HalfUnitF4),
        Unif: McTolerance(drawsUnif[i], HalfUnitF3),
        Jeff: McTolerance(drawsJeff[i], HalfUnitF3));
}).ToArray();
double medianUnifTolerance = McTolerance(medianDrawsUnif, HalfUnitF3);
double medianJeffTolerance = McTolerance(medianDrawsJeff, HalfUnitF3);
double minUnifTolerance = McTolerance(minDrawsUnif, HalfUnitF3);
double minJeffTolerance = McTolerance(minDrawsJeff, HalfUnitF3);

// Mesure C# : deux tirages successifs par couple p,N, comme dans le protocole Python.
var rows = instances.Select(i =>
{
    var unif = BayesianVsFrequentistWidth(i.N, i.P, 1.0, 1.0, CredLevel);
    var jeff = BayesianVsFrequentistWidth(i.N, i.P, 0.5, 0.5, CredLevel);
    return (P: i.P, N: i.N, HdiUnif: unif.HdiWidth, Hoeff: unif.Hoeffding, RUnif: unif.Ratio, RJeff: jeff.Ratio);
}).ToArray();
WriteInvariant($"{"p",5} {"N",5} {"HDI_unif",10} {"Hoeff",10} {"R_unif",8} {"R_jeff",8}");
foreach (var row in rows)
    WriteInvariant($"{row.P,5:F2} {row.N,5:D} {row.HdiUnif,10:F4} {row.Hoeff,10:F4} {row.RUnif,8:F3} {row.RJeff,8:F3}");

// Comparaison exécutée ligne par ligne : égalité de la grille, parité de la
// formule fermée, écarts stochastiques et ordre strict pour chaque a priori.
double gridHoeffError = 0.0;
double gridHdiError = 0.0;
double gridRatioError = 0.0;
int passed = 0;
for (int i = 0; i < rows.Length; i++)
{
    var row = rows[i];
    var reference = cited[i];
    string label = FormattableString.Invariant($"p={row.P:F2}, N={row.N}");
    Require(row.P == reference.P && row.N == reference.N,
        $"{label} : couple différent de la référence citée p={reference.P:F2}, N={reference.N}.");
    Require(row.RUnif > 0.0 && row.RUnif < 1.0, $"{label} : ratio uniforme hors de ]0, 1[.");
    Require(row.RJeff > 0.0 && row.RJeff < 1.0, $"{label} : ratio Jeffreys hors de ]0, 1[.");
    passed++;
    WriteInvariant($"{label} : ratios {row.RUnif:F3} et {row.RJeff:F3} < 1 — CONFORME");
    gridHoeffError = Math.Max(gridHoeffError, CheckClose(label + " — Hoeff", row.Hoeff, reference.Hoeff, HalfUnitF4));
    gridHdiError = Math.Max(gridHdiError, CheckClose(label + " — HDI uniforme", row.HdiUnif, reference.HdiUnif, tolerances[i].Hdi));
    gridRatioError = Math.Max(gridRatioError, CheckClose(label + " — ratio uniforme", row.RUnif, reference.RUnif, tolerances[i].Unif));
    gridRatioError = Math.Max(gridRatioError, CheckClose(label + " — ratio Jeffreys", row.RJeff, reference.RJeff, tolerances[i].Jeff));
}

var ratiosUnif = rows.Select(r => r.RUnif).ToArray();
var ratiosJeff = rows.Select(r => r.RJeff).ToArray();
double medianUnif = Median(ratiosUnif);
double medianJeff = Median(ratiosJeff);
var minUnifRow = rows.OrderBy(r => r.RUnif).First();
var minJeffRow = rows.OrderBy(r => r.RJeff).First();

// Références CITÉES : médianes et minima affichés sous la table Python.
double aggregateError = new[]
{
    CheckClose("Médiane uniforme", medianUnif, 0.585, medianUnifTolerance),
    CheckClose("Minimum uniforme", minUnifRow.RUnif, 0.291, minUnifTolerance),
    CheckClose("Médiane Jeffreys", medianJeff, 0.591, medianJeffTolerance),
    CheckClose("Minimum Jeffreys", minJeffRow.RJeff, 0.309, minJeffTolerance)
}.Max();

// Tableau de lecture : toutes les valeurs ci-dessous proviennent de la mesure C#.
Console.WriteLine();
Console.WriteLine("Résultats C# — mesure sur 20 instances BSC");
Console.WriteLine("A priori                 Médiane   Minimum    (p, N) du minimum");
WriteInvariant($"Beta(1,1) uniforme        {medianUnif:F3}     {minUnifRow.RUnif:F3}      ({minUnifRow.P:F2}, {minUnifRow.N})");
WriteInvariant($"Beta(0.5,0.5) Jeffreys    {medianJeff:F3}     {minJeffRow.RJeff:F3}      ({minJeffRow.P:F2}, {minJeffRow.N})");
WriteInvariant($"Réduction médiane des largeurs : {1.0 - medianUnif:P0} (uniforme), {1.0 - medianJeff:P0} (Jeffreys).");
foreach (var group in rows.GroupBy(r => r.P))
    WriteInvariant($"p={group.Key:F2} : ratios de {group.Min(r => Math.Min(r.RUnif, r.RJeff)):F3} à {group.Max(r => Math.Max(r.RUnif, r.RJeff)):F3} (deux a priori).");
foreach (var group in rows.GroupBy(r => r.N))
    WriteInvariant($"N={group.Key} : médianes {Median(group.Select(r => r.RUnif)):F3} (uniforme) et {Median(group.Select(r => r.RJeff)):F3} (Jeffreys).");
var largestN = rows.Where(r => r.N == ns.Max()).ToArray();
WriteInvariant($"À N={ns.Max()} : ratios de {largestN.Min(r => Math.Min(r.RUnif, r.RJeff)):F3} à {largestN.Max(r => Math.Max(r.RUnif, r.RJeff)):F3}.");

// Attestation : écarts absolus mesurés aux sorties citées, par section.
Console.WriteLine("\nAttestation de parité — écarts absolus mesurés");
WriteInvariant($"Posterior : max={posteriorMaxError:E3} ; HDI déterministe : max={hdiMaxError:E3}.");
WriteInvariant($"Exemple 3 : Hoeffding={smokeHoeffError:E3}, largeur HDI={smokeHdiError:E3}, ratio={smokeRatioError:E3}.");
WriteInvariant($"Grille BSC : Hoeffding max={gridHoeffError:E3}, largeur HDI uniforme max={gridHdiError:E3}, ratios max={gridRatioError:E3}, agrégats max={aggregateError:E3}.");
Require(passed == ExpectedInstances, $"Invariant validé sur {passed} couples au lieu de {ExpectedInstances}.");
WriteInvariant($"Ratios inférieurs à 1 pour les deux a priori : {passed}/{ExpectedInstances} instances — CONFORME.");

    p     N   HDI_unif      Hoeff   R_unif   R_jeff


 0.05   100     0.1350     0.2716    0.497    0.363


 0.05   500     0.0405     0.1215    0.333    0.309


 0.05  1000     0.0300     0.0859    0.349    0.314


 0.05  5000     0.0125     0.0384    0.325    0.325


 0.10   100     0.1125     0.2716    0.414    0.495


... (135 outputs left out)

Posterior : max=0.000E+000 ; HDI déterministe : max=2.605E-005.


Exemple 3 : Hoeffding=2.030E-005, largeur HDI=7.500E-003, ratio=2.782E-002.


Grille BSC : Hoeffding max=2.771E-005, largeur HDI uniforme max=3.450E-002, ratios max=1.270E-001, agrégats max=3.441E-002.


Ratios inférieurs à 1 pour les deux a priori : 20/20 instances — CONFORME.


## Résultats — mesure sur 20 instances BSC

Mesure effectuée en C# (20 instances BSC, seed=42, p ∈ {0.05, 0.1, 0.2, 0.3, 0.5}, N ∈ {100, 500, 1000, 5000} ; mêmes seed/params que Lean-19/19b/19c) :

| A priori | Ratio médian HDI/Hoeffding | Ratio minimal HDI/Hoeffding | Couple (p, N) du minimum |
|---|---:|---:|---:|
| Beta(1, 1) uniforme | 0.576 | 0.325 | (0.05, 5000) |
| Beta(0.5, 0.5) Jeffreys | 0.580 | 0.309 | (0.05, 500) |

**Lecture** : le HDI bayésien est **plus serré** que Hoeffding sur **toutes** les 20 instances, pour les deux a priori. La réduction médiane des largeurs est chiffrée dans la sortie C#. À `N=5000`, l'écart reste visible ; la grille ne démontre pas à elle seule un résultat asymptotique. **Hoeffding est une borne p-free, pas un estimateur** ; le HDI paramétrique hérite de la `p(1-p)` du posterior.

**Décomposition par p** :
- Pour `p=0.05` (rares succès), lire la plage des ratios en tête de la décomposition. Comparer les deux colonnes du tableau : elles ne reposent pas sur le même tirage, donc leur écart ne mesure pas isolément l'effet de l'a priori.
- Pour `p=0.5`, la variance `p(1-p) = 0.25` est maximale : c'est dans ce groupe que le tableau C# donne les ratios les plus proches de 1.
- Pour `p ∈ {0.1, 0.2, 0.3}`, les plages affichées se situent entre celles des deux groupes extrêmes.

**Décomposition par N** : les médianes calculées pour `N=100`, `N=500`, `N=1000` et `N=5000` figurent sous le tableau C#. Pour chaque taille, elles restent inférieures à 1.

**Comparaison avec le trio Lean-19/19b/19c** : HDI bayésien et Chernoff exploitent tous deux la variance `p(1-p)`. Hoeffding, distribution-free, est la référence conservative.

## Comparaison avec le trio Lean-19/19b/19c

| Carnet | Méthode | Hypothèse | Pour N petit | Pour N grand |
|--------|---------|-----------|--------------|--------------|
| Lean-19 | Hoeffding | distribution-free | OK (référence) | OK (référence) |
| Lean-19b | Chernoff | mgf exacte Bernoulli | **plus tight** | rejoint Hoeffding |
| Lean-19c | Bernstein | variance réelle p(1-p) | intermédiaire | intermédiaire |
| **Lean-19d** | **Beta-Binomial** | **prior conjugué** | **plus serré sur la grille** | **plus serré jusqu'à N=5000 sur la grille** |

**Note pédagogique** : Hoeffding borne `|p_hat - p|` uniformément sur `[0,1]`, tandis que le HDI exploite la forme du posterior. La comparaison porte sur leurs largeurs, pas sur une équivalence de leurs garanties.

## Exercices

Les trois exercices portent sur les memes objets que les exemples guides ci-dessus :
posterior conjugue, intervalle de credibilite, confrontation au frequentiste. Ils se
resolvent avec les fonctions deja definies dans ce carnet (`BetaPosteriorParams`,
`BetaHdi`, `HoeffdingWidth`) et le generateur `rng` (seed=42, commun a la famille).

**E1.** Implementez `PosteriorMeanExo(priorAlpha, priorBeta, successes, failures)` qui
retourne la moyenne du posterior Beta-Binomial.

**E2.** Implementez `HdiRatioExo(priorAlpha, priorBeta, n, p, credLevel = 0.95)` qui
retourne le rapport `largeur HDI / largeur Hoeffding` pour une instance simulee.

**E3.** Implementez `SweepCredLevelsExo(priorAlpha, priorBeta, n, p, levels)` qui
retourne un dictionnaire `{niveau: ratio}` en iterant sur les niveaux de credibilite.
Les niveaux par défaut sont `0.80`, `0.90`, `0.95` et `0.99`.

> Ce carnet reprend le parcours de la [version Python](Lean-19d-Inference-Bayesian-Beta-Binomial-Python.ipynb).

In [6]:
// Illustration 1 — Moyenne a posteriori
 double PosteriorMeanExo(double priorAlpha, double priorBeta, int successes, int failures)
{
    // TODO etudiant
    return default;
}

In [7]:
// Illustration 2 — Rapport des largeurs
double HdiRatioExo(double priorAlpha, double priorBeta, int n, double p, double credLevel = 0.95)
{
    // TODO etudiant : rapport largeur HDI / largeur Hoeffding pour l'instance (n, p).
    return default;
}

In [8]:
// Illustration 3 — Balayage des niveaux de crédibilité
Dictionary<double, double> SweepCredLevelsExo(double priorAlpha, double priorBeta, int n, double p, IReadOnlyList<double> levels)
{
    // TODO etudiant : retourner un dictionnaire qui associe chaque niveau à son ratio.
    return new Dictionary<double, double>();
}

// Surcharge avec les niveaux de crédibilité par défaut.
Dictionary<double, double> SweepCredLevelsExo(double priorAlpha, double priorBeta, int n, double p) =>
    SweepCredLevelsExo(priorAlpha, priorBeta, n, p, new[] { 0.80, 0.90, 0.95, 0.99 });